In [ ]:
import os
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

def extract_entities_labels(line):
    try:
        triples_part = line.split('<|triple|>')[1].split('<|endoftriple|>')[0]
        entity_parts = triples_part.split('[')
        entity_labels = []
        for entity_part in entity_parts:
            if 'entity' in entity_part:
                label = entity_part.split(']:')[1].split(',')[0].strip().rstrip('}')
                entity_labels.append(label)
        return entity_labels[:2]  # Return only the first two entity labels
    except IndexError:
        return ["Unknown", "Unknown"]

def build_entity_to_lines_map(hops_data):
    entity_to_lines = {}
    for entry in hops_data:
        hop = entry['hop']
        first_entity = hop.split('[')[1].split(']:')[1].split(',')[0].strip()
        if first_entity in entity_to_lines:
            entity_to_lines[first_entity].append(hop)
        else:
            entity_to_lines[first_entity] = [hop]
    return entity_to_lines

def find_matching_lines(entity, entity_to_lines):
    return entity_to_lines.get(entity, [])

def process_files(merged_file, hops_file):
    # Load merged.txt file
    with open(merged_file, 'r') as file:
        merged_data = file.readlines()
    
    # Extract entity labels from merged.txt
    entity_labels = [extract_entities_labels(line) for line in merged_data]
    
    # Load hops.jsonl file
    with open(hops_file, 'r') as file:
        hops_data = [json.loads(line) for line in file]
    
    # Build entity to lines map for fast lookup
    entity_to_lines = build_entity_to_lines_map(hops_data)
    
    # Append the matching lines from hops file to the corresponding lines in merged file
    for i, labels in enumerate(entity_labels):
        if labels[0] != "Unknown" and labels[1] != "Unknown":
            line1_contents = find_matching_lines(labels[0], entity_to_lines)
            line2_contents = find_matching_lines(labels[1], entity_to_lines)
            all_lines = line1_contents + line2_contents
            merged_data[i] = merged_data[i].strip() + ' ' + ' '.join(all_lines) + '\n'
    
    return merged_data

def save_results(merged_file, hops_file):
    merged_file_path = os.path.join(merged_files_dir, merged_file)
    hops_file_path = os.path.join(hops_files_dir, hops_file)
    
    result_data = process_files(merged_file_path, hops_file_path)
    
    # Save the result to a new file in the output directory
    output_file = merged_file.replace('merged', 'output')
    with open(os.path.join(output_dir, output_file), 'w') as file:
        file.writelines(result_data)

# Directory containing the files
merged_files_dir = "newtext2kg"
hops_files_dir = "generated_hops"
output_dir = "0"

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Get list of files
merged_files = sorted([f for f in os.listdir(merged_files_dir) if f.startswith('merged') and f.endswith('.txt')])
hops_files = sorted([f for f in os.listdir(hops_files_dir) if f.endswith('.jsonl') and '__hops' in f])

# Pair files based on numbering
file_pairs = []
for merged_file in merged_files:
    number = merged_file.split('_')[1].split('.')[0]  # Extract the number
    hops_file = f"{number}__hops.jsonl"
    if hops_file in hops_files:
        file_pairs.append((merged_file, hops_file))

# Define the number of threads
num_threads = 152

# Process each pair of files with a progress bar and multithreading
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    list(tqdm(executor.map(lambda pair: save_results(pair[0], pair[1]), file_pairs), total=len(file_pairs), desc="Processing files"))

print("Batch processing completed.")


In [ ]:
import os
from tqdm import tqdm

folder_path = '0'

txt_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]

for filename in tqdm(txt_files, desc="Processing files"):
    file_path = os.path.join(folder_path, filename)
    
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    modified_lines = []
    for line in lines:
        modified_line = line.replace('} <|endofhops|><|sentence|>', '<|sentence|>')
        modified_lines.append(modified_line)
    
    with open(file_path, 'w', encoding='utf-8') as file:
        file.writelines(modified_lines)

print("finish")


In [ ]:
import os
from tqdm import tqdm

folder_path = '0'

txt_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]

for txt_file in tqdm(txt_files, desc="Processing files"):
    file_path = os.path.join(folder_path, txt_file)
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    with open(file_path, 'w', encoding='utf-8') as file:
        for line in lines:
            # replace ' hops: ' with '}   {'
            line = line.replace(' hops: ', '} <|endofhop|> <|hop|> {')
            # add '}  '
            line = line.rstrip('\n') + '} <|endofhop|> <|endofhops|>\n'
            file.write(line)


In [ ]:
import os
import re
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

def unicode_to_text(unicode_str):
    try:
        return unicode_str.encode('latin1').decode('unicode_escape')
    except UnicodeDecodeError:
        return unicode_str

def process_file(file_path):
    new_lines = []
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        for line in file:
            unicode_matches = re.findall(r'\\u[0-9a-fA-F]{4}', line)
            for match in unicode_matches:
                line = line.replace(match, unicode_to_text(match))
            new_lines.append(line)
    return new_lines

def save_file(file_path, content):
    with open(file_path, 'w', encoding='utf-8', errors='ignore') as file:
        file.writelines(content)

def process_and_save_file(file_path):
    processed_content = process_file(file_path)
    save_file(file_path, processed_content)

def main(folder_path, max_workers=4):
    txt_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.txt')]
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(tqdm(executor.map(process_and_save_file, txt_files), total=len(txt_files), desc="Processing files"))

if __name__ == "__main__":
    folder_path = "2"  
    main(folder_path)


In [ ]:
import os

def remove_last_line_from_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    if lines:
        with open(filepath, 'w', encoding='utf-8') as file:
            file.writelines(lines[:-1])

def remove_last_line_from_files_in_folder(folder_path):
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            filepath = os.path.join(folder_path, filename)
            remove_last_line_from_file(filepath)

folder_path = '2'  
remove_last_line_from_files_in_folder(folder_path)
